In [2]:
import math
import time
from enum import Enum, auto
# my utils
from utils import dotdict

import cv2
import numpy as np
from scipy.spatial.transform import Rotation 

# Mac screen info
from AppKit import NSScreen
SCREEN_WIDTH, SCREEN_HIGHT = NSScreen.screens()[0].frame().size.width, NSScreen.screens()[0].frame().size.height

# various filter: https://filterpy.readthedocs.io/en/latest/
from filterpy.gh import GHFilter
from filterpy.kalman import KalmanFilter
from filterpy.common import Q_discrete_white_noise

# how to control your mouse: https://stackoverflow.com/questions/281133/how-to-control-the-mouse-in-mac-using-python
# mouse controll: https://pypi.org/project/pynput/
from pynput import mouse as Mouse

# hand landmark: https://google.github.io/mediapipe/solutions/hands.html
import mediapipe as mp 
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

In [3]:
# def print_coordinate(hand_landmark):
  # finger = ['Index_finger', 'Middle_finger', 'Ring_finger', 'Pinky']
  # position = ['mcp', 'pip', 'dip', 'tip']

  # if hand_landmark == mp_hands.HandLandmark.WRIST:
    # name = 'Wrist'
  # elif 1 <= hand_landmark <= 4:
    # name = f"{'Thumb'} {['cmc', 'mcp', 'ip', 'tip'][(hand_landmark-1)%4]}"
  # elif 5 <= hand_landmark <= 20:
    # name = f'{finger[(hand_landmark-1)//4]} {position[(hand_landmark-1)%4]}'
  # else:
    # raise 'Unknow hand landmark'

  # print(
      # f'{name} coordinate: (',
      # f'{raw_landmarks.landmark[hand_landmark].x * image_width :.3f}, '
      # f'{raw_landmarks.landmark[hand_landmark].y * image_width :.3f}, '
      # f'{raw_landmarks.landmark[hand_landmark].z * image_width :.3f}, '
      # f')'
  # )

In [4]:
def draw_handedness(img, handedness):
  if handedness[0] < 0.5:
    label = 'Left' 
  elif handedness[0] > 0.5:
    label = 'Right' 
  else:
    label = 'Unknown'

  img_w, img_h = img.shape[1], img.shape[0]
  cv2.putText(img, f'{label} {np.abs((handedness[0]-.5)*2) :.2f}',
      org=(img_w//2 - 200, 30), # LeftBottomCornerOfText
      fontFace=cv2.FONT_HERSHEY_SIMPLEX, 
      fontScale=1,
      color=(0, 0, 255),
      lineType=2)

def draw_landmarks(img, landmarks, color=(50, 255, 50)):
  img_w, img_h = img.shape[1], img.shape[0]
  # draw dot
  for i in range(21):
    p = landmarks[i, 0:2, 0]
    cv2.circle(img, p.astype(int), radius=4, thickness=-1, color=(50, 50, 255))

  # draw line 
  pairs = []
  # wrist to index~pinky
  pairs.append((0, 1))
  pairs.append((0, 5))
  pairs.append((0, 17))
  # finger to finger
  pairs.append((5, 9))
  pairs.append((9, 13))
  pairs.append((13, 17))
  # mcp to tip
  for i in range(1, 21, 4):
    for j in range(3):
      pairs.append((i+j, i+j+1))

  for pp1, pp2 in pairs:
    p1 = landmarks[pp1, 0:2, 0]
    p2 = landmarks[pp2, 0:2, 0]
    cv2.line(img, p1.astype(int), p2.astype(int), thickness=2, color=color)

def draw_finger_state(img, handedness, finger_states):
  img_w, img_h = img.shape[1], img.shape[0]

  state_str = ''
  for finger_idx in range(5): 
    if handedness[0] < 0.5:
      state_str = f'{finger_states[finger_idx].value}{state_str}'
    else:
      state_str = f'{state_str}{finger_states[finger_idx].value}'      
  cv2.putText(img, state_str,
      org=(img_w//2, 30), # bottomLeftCornerOfText
      fontFace=cv2.FONT_HERSHEY_SIMPLEX, 
      fontScale=1,
      color=(0, 0, 255),
      lineType=2)


In [5]:
DESIRED_HEIGHT = 720
DESIRED_WIDTH = 720
def preprocess_img(image):
  h, w = image.shape[:2]
  # resize
  if h < w:
    img = cv2.resize(image, (DESIRED_WIDTH, math.floor(h/(w/DESIRED_WIDTH))))
  else:
    img = cv2.resize(image, (math.floor(w/(h/DESIRED_HEIGHT)), DESIRED_HEIGHT))

  return img

In [6]:
def landmarks2vec(hand_landmarks):
  vecs = np.empty((21, 3))
  for landmark_idx in mp_hands.HandLandmark:
    vecs[landmark_idx] = np.array([
      hand_landmarks.landmark[landmark_idx].x,
      hand_landmarks.landmark[landmark_idx].y,
      hand_landmarks.landmark[landmark_idx].z,
    ])
  return vecs

![21 hand landmarks](https://google.github.io/mediapipe/images/mobile/hand_landmarks.png)

## Spec

### Tracking
- existence
  - shape: (1,)
  - range: [0, 1]
- handedness
  - shape: (1,)
  - range: [0, 1] (0: left, 1: right)
- landmarks
  - shape: (21, 3,)
  - range: [0, 1] (scale with `image_width`, `image_height`, `image_width`)

### Filter
- existence
  - *state shape: (2,) (position, velocity)*
  - *z shape: (1,) (position)*
  - x: [0.5, 0].T
  - F: [[1, dt], [0, 1]]
  - H: [1, 0]
  - init_P: eye(2,2) * 0.5
  - init_Q: eye(2,2) * 0 **(not sure)**
  - init_R: 0.2
- handedness 
  - *state shape: (2,) (position, velocity)*
  - *z shape: (1,) (position)*
  - x: [0.5, 0].T
  - F: [[1, dt], [0, 1]]
  - H: [1, 0]
  - init_P: eye(2,2) * 0.5
  - init_Q: eye(2,2) * 0 **(not sure)**
  - init_R: 0.5 **(tuning)**
- landmarks 
  - *state shape: (21 * 3 * 2,) = (126,) ((landmarks) (x,y,z) (position, velocity))*
  - *z shape: (21 * 3,) = (63,) ((landmarks) (x,y,z))*
  - x: [n=126, value=0.5].T
  - F: [[1, dt, 0, ...], [0, 1, dt, 0, ...], [0, 0, 1, dt, 0, ...] ... [dt, 0, ..., 1]]
  - H: [1, 0, 1, 0, ...]
  - init_P: eye(126,126) * 0.5
  - init_Q: eye(126,126) * 0 **(not sure)**
  - init_R: 0.01 **(should be low)**

In [7]:
def block_diagonal_array(n, block):
  """
  Returns block diagonal array
  n: # of block
  block: block array
  """
  block = np.array(block)
  arr = None
  for i in range(n):
    tmp = np.array(block)
    tmp = np.hstack([
            np.zeros((block.shape[0], block.shape[1]*i)),
            tmp,
            np.zeros((block.shape[0], block.shape[1]*(n-1-i))),
          ])
    
    if arr is None:
      arr = tmp
    else:
      arr = np.vstack([arr, tmp])
  return arr

In [8]:
def angle_between_vectors(v1, v2):
  unit_v1 = v1 / np.linalg.norm(v1)
  unit_v2 = v2 / np.linalg.norm(v2)
  dot_product = np.dot(unit_v1, unit_v2)
  # angle in radien
  angle = np.arccos(dot_product)
  return angle

def orthogonal_projection(v1, v2):
  """
  project v1 onto v2 plane
  return orthogonal_vector , orthogonal_complement
  https://ccjou.wordpress.com/2010/04/19/%E6%AD%A3%E4%BA%A4%E6%8A%95%E5%BD%B1-%E5%A8%81%E5%8A%9B%E5%BC%B7%E5%A4%A7%E7%9A%84%E4%BB%A3%E6%95%B8%E5%B7%A5%E5%85%B7/
  """
  assert len(v1.shape) == 1
  assert len(v2.shape) == 1

  z = np.array(v1).reshape(-1,1)
  x = np.array(v2).reshape(-1,1)
  P = x @ x.T / (x.T @ x) 
  L = np.eye(x.shape[0]) - P
  return L @ z, P @ z

In [134]:
def rotate_coordinate_matrix(new_axis, old_axis):
  """
  Calculate rotation matrix for rotating from old-axis to new_axis
  new_axis: [new_axis_1, new_axis_2]  
  old_axis: [old_axis_1, old_axis_2]  
  return: rotation matrix, inverse of rotation matrix
  """
  assert np.array(new_axis).shape == (2, 3)
  assert np.array(old_axis).shape == (2, 3)
  assert np.isclose(np.dot(new_axis[0], new_axis[1]), 0)
  assert np.isclose(np.dot(old_axis[0], old_axis[1]), 0)

  new_x = np.array(new_axis[0]) / np.linalg.norm(new_axis[0])
  old_x = np.array(old_axis[0]) / np.linalg.norm(old_axis[0])
  new_y = np.array(new_axis[1]) / np.linalg.norm(new_axis[1])
  old_y = np.array(old_axis[1]) / np.linalg.norm(old_axis[1])

  # rotate x-axis
  cross_x = np.cross(old_x, new_x)
  cross_x = cross_x / np.linalg.norm(cross_x)
  theta_x = angle_between_vectors(old_x, new_x)

  r_x = Rotation.from_rotvec(cross_x * theta_x)

  old_y2 = old_y @ r1.as_matrix()

  cross_y = np.cross(old_y2, new_y)
  cross_y = cross_y / np.linalg.norm(cross_y)
  theta_y = angle_between_vectors(old_y2, new_y)

  r_y = Rotation.from_rotvec(cross_b * theta_b)

  r = r1.as_matrix() @ r2.as_matrix()

  return r, r.T

In [9]:
def switch_hand(landmarks):
  res = np.array(landmarks)
  res[[1, 2, 3, 4, 5, 6, 7, 8]] = res[[17, 18, 19, 20, 13, 14, 15, 16]] 
  return res

In [10]:
def pos_vel_filter(x, P, R, Q=0., dt=1.):
  """ 
  Returns a KalmanFilter which implments a 
  constant velocity model for a state [x dx].T
  """
  kf = KalmanFilter(dim_x=2, dim_z=1)
  kf.x = np.array([x[0], x[1]]) # position and velocity
  kf.F = np.array([[1., dt],     # state transition matrix
                  [0., 1.]])
  kf.H = np.array([[1., 0]])    # measurement function

  kf.R *= R                     # measurement uncertainty
  if np.isscalar(P):
    kf.P *= P                   # covariance matrix
  else:
    kf.P[:] = P                 # [:] makes deep copy
  if np.isscalar(Q):
    kf.Q *= Q_discrete_white_noise(dim=2, dt=dt, var=Q)
  else:
    kf.Q[:] = Q

  return kf

In [11]:
def multi_pos_vel_filter(x, P, R, Q=0., dt=1.):
  """ 
  Returns a KalmanFilter which implments a 
  constant velocity model for # of dim states.
  """
  dim = len(x) // 2
  kf = KalmanFilter(dim_x=dim*2, dim_z=dim)
  kf.x = np.array(x) # position and velocity
  kf.F = block_diagonal_array(dim, [[1., dt], [0., 1.]])  # state transition matrix
  kf.H = block_diagonal_array(dim, [[1, 0]])    # measurement function

  kf.R *= R                     # measurement uncertainty
  if np.isscalar(P):
    kf.P *= P                   # covariance matrix
  else:
    kf.P[:] = P                 # [:] makes deep copy
  if np.isscalar(Q):
    kf.Q *= Q_discrete_white_noise(dim=2, dt=dt, var=Q, block_size=dim)
  else:
    kf.Q[:] = Q

  return kf

In [12]:
class FINGER_STATE(Enum):
  BENT = 0
  STRAIGHT = 1
  UNKNOWN = auto()

class FINGER(Enum):
  THUMB = 0
  INDEX = 1
  MIDDLE = 2
  RING = 3
  PINKY = 4


def get_finger_state(landmarks):
  res = np.array([FINGER_STATE.UNKNOWN for i in range(5)])
  
  # finger_idx
  # thumb: 1~4, index: 5~8, middle: 9~12, ring: 13~16, pinky: 17~20
  for finger_idx in range(1, 21, 4):
    tip = landmarks[finger_idx+3, :, 0]
    dip = landmarks[finger_idx+2, :, 0]
    pip = landmarks[finger_idx+1, :, 0]
    mcp = landmarks[finger_idx, :, 0]

    # print(f'{finger_name[(finger_idx-1)//4]} finger')
    # print(f'{np.pi - angle_between_vectors(pip - mcp, pip - dip)}', end='')
    # print(f'{np.pi - angle_between_vectors(dip - pip, dip - tip)}')
    

    accumulated_angle = (np.pi - angle_between_vectors(pip - mcp, pip - dip)) + (np.pi - angle_between_vectors(dip - pip, dip - tip))
    
    if accumulated_angle > (np.pi * 0.4):
      res[(finger_idx-1) // 4] = FINGER_STATE.BENT
    else:
      res[(finger_idx-1) // 4] = FINGER_STATE.STRAIGHT

  return res

In [140]:
mouse = Mouse.Controller()
# mouse_listener = Mouse.Listener(
  # on_move=lambda x,y: print(f'Pointer move to ({x :.2f}, {y :.2f})'),
  # on_scroll=lambda x,y,dx,dy: print(f'Scrolled ({x :.2f}, {y :.2f}) at {"down" if dy < 0 else "up"}'))
# mouse_listener.start()
# mouse_listener.wait()


frame_cnt = 0
prev_frame_cnt = 0
prev_timestamp = time.time()

# z_handedness, z_landmarks = dotdict({'label': 'none', 'score': 0, 'index': 0.5}), np.zeros((21, 3)) 

# init estimate
existence = [.5, 0]
handedness = [.5, 0]
landmarks = np.full((21, 3, 2), 0.5)

# filter
dt = 1.
existence_f = pos_vel_filter(existence, P=.5, R=.5, Q=.001, dt=dt)
handedness_f = pos_vel_filter(handedness, P=.5, R=.1, Q=.1, dt=dt)
landmarks_f = multi_pos_vel_filter(landmarks.flatten(), P=DESIRED_WIDTH/3., R=50, Q=40, dt=dt)

# gesture
index_tips = np.array([])


# For webcam input:
cap = cv2.VideoCapture(0)
with mp_hands.Hands(
    min_detection_confidence=0.75,
    min_tracking_confidence=0.7) as hands:
  while cap.isOpened():
    success, raw_image = cap.read()
    if not success:
      print("Ignoring empty camera frame.")
      # If loading a video, use 'break' instead of 'continue'.
      continue

    # Flip the image horizontally for a later selfie-view display, and convert the BGR image to RGB.
    image = cv2.cvtColor(cv2.flip(preprocess_img(raw_image), 1), cv2.COLOR_BGR2RGB)
    # To improve performance, optionally mark the image as not writeable to pass by reference.
    image.flags.writeable = False
    results = hands.process(image)
    
    image_hight, image_width, _ = image.shape
    # Draw the hand annotations on the image.
    image.flags.writeable = True
    annotated_image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)


    # existence detection
    if results.multi_hand_landmarks:
      z_existence = 1
    else :
      z_existence = 0
    existence_f.predict()
    existence_f.update(z_existence)
    existence = existence_f.x
      

    if existence[0] > 0.5:

      handedness_f.predict()
      landmarks_f.predict()

      if z_existence == 1:
        # NOTE: `existence` only response for one hand existence
        raw_handedness, raw_landmarks = results.multi_handedness[0], results.multi_hand_landmarks[0]

        z_handedness = .5 + raw_handedness.classification[0].score * (.5 if raw_handedness.classification[0].index == 1 else -.5)
        z_landmarks = landmarks2vec(raw_landmarks)
        # scale to image size
        z_landmarks *= [image_width, image_hight, image_width] 
        # rotate to palm surface
        axis_y = z_landmarks[9] - z_landmarks[0]
        axis_z = np.cross(z_landmarks[17] - z_landmarks[0], axis_y)
        r, inv_r = rotate_coordinate_matrix([axis_y, axis_z], [[0,1,0], [0,0,1]])
        z_landmarks = z_landmarks @ r

        handedness_f.update(z_handedness)

        # switch landmarks
        should_switch_hand = (handedness[0] > 0.5 and z_handedness < 0.5) or (handedness[0] < 0.5 and z_handedness > 0.5)
        landmarks_f.update(switch_hand(z_landmarks).flatten() if should_switch_hand else z_landmarks.flatten())

      handedness = handedness_f.x
      landmarks = landmarks_f.x.reshape(21, 3, 2)


      # Gesture
      # TODO: project landmarks on to palm surface
      finger_states = get_finger_state(landmarks)

      ## move mouse
      if np.all(finger_states[[2,3]] == FINGER_STATE.BENT) \
          and (
            (np.all(np.abs(landmarks[0, :, 1]) < 4.) and np.any(np.abs(landmarks[8, 0:2, 1] > 1.)))
            or (finger_states[1] == FINGER_STATE.STRAIGHT)
          ):
          # and np.all(finger_states[1] == FINGER_STATE.STRAIGHT) \
        x = np.clip(landmarks[8, 0, 1] * 15, -100, 100) 
        y = np.clip(landmarks[8, 1, 1] * 15, -100, 100) 

        # WARN: mouse may move to the negative position, which refer to second monitor 
        # clip mouse position in the main screen
        x = np.clip(x, 0 - mouse.position[0], SCREEN_WIDTH - mouse.position[0])
        y = np.clip(y, 0 - mouse.position[1], SCREEN_HIGHT - mouse.position[1])
        mouse.move(x, y)
        # print(f'index tip vel {landmarks[8, 0:2, 1]}')

      # ## scroll vertical
      # # TODO: smooth scroll (keep scroll after gesture disapper)
      # # TODO: condition by bent and wrist movement being small, scrolling finger do not to be straight

      # # scroll up
      # if np.all([finger_states[i] == FINGER_STATE.BENT for i in [4]]) \
          # and np.all([finger_states[i] == FINGER_STATE.STRAIGHT for i in [1,2,3]]):
        # if index_tip_vy < 0:
          # mouse.scroll(0, -index_tip_vy*0.075)
          # cv2.line(annotated_image, index_tip[0:2].astype(int), (index_tip[0:2]+[0, index_tip_vy]).astype(int), color=(50, 50, 255), thickness=3)

      # # scroll down
      # if np.all([finger_states[i] == FINGER_STATE.BENT for i in [3,4]]) \
          # and np.all([finger_states[i] == FINGER_STATE.STRAIGHT for i in [1,2]]):
        # if index_tip_vy < 0:
          # mouse.scroll(0, index_tip_vy*0.075)
          # cv2.line(annotated_image, index_tip[0:2].astype(int), (index_tip[0:2]+[0, -index_tip_vy]).astype(int), color=(255, 50, 50), thickness=3)


      ## switch desktop
      ## switch screen
      
      # update prev value
      # prev_index_tip = np.array([index_tip_x, index_tip_y, index_tip[2]])



      # Draw info
      # measurement landmarks
      # mp_drawing.draw_landmarks(annotated_image, raw_landmarks, mp_hands.HAND_CONNECTIONS)

      inv_landmarks = np.stack([landmarks[:, :, 0] @ inv_r, landmarks[:, :, 1] @ inv_r], axis=2)
      draw_handedness(annotated_image, handedness)
      # draw_landmarks(annotated_image, np.stack([z_landmarks, np.zeros(z_landmarks.shape)], axis=2), (255, 50, 50, 0.5))
      draw_landmarks(annotated_image, inv_landmarks)
      draw_finger_state(annotated_image, handedness, finger_states)

      cv2.putText(annotated_image, f'{landmarks[8, 0, 0] :.2f}, {landmarks[8, 0, 1] :.2f}',
          org=(image_width*0, image_hight//2), # bottomLeftCornerOfText
          fontFace=cv2.FONT_HERSHEY_SIMPLEX, 
          fontScale=1,
          color=(0, 0, 255),
          lineType=2)

      # Print info
      # print(
        # state_str, 
        # handedness.label, 
        # f'{handedness.score :.2f}',
        # f'{raw_handedness.classification[0].label}',
        # f'{raw_handedness.classification[0].score :.2f}',
        # f'{[np.round(i, 2) for i in landmarks[mp_hands.HandLandmark.INDEX_FINGER_TIP]]}',
      # )

    else: 
      pass
      # print('-------------- no hand found --------------')


    
    # Draw fps
    frame_cnt += 1
    now_timestamp = time.time()
    if now_timestamp - prev_timestamp >= 1:
      prev_timestamp = now_timestamp
      prev_frame_cnt = frame_cnt
      frame_cnt = 0
    cv2.putText(annotated_image, f'{prev_frame_cnt}',
        org=(DESIRED_HEIGHT - 30, 20), # bottomLeftCornerOfText
        fontFace=cv2.FONT_HERSHEY_SIMPLEX, 
        fontScale=0.5,
        color=(100, 255, 100),
        lineType=2)


    cv2.imshow('Hands', annotated_image)
    if cv2.waitKey(20) & 0xFF == 27:
      # FIXME: `Listener.stop()` failed to stop mouse listerner
      # mouse_listener.stop()
      break

cap.release()
# cv2.destroyAllWindows()

In [89]:
# from
a1 = np.array([1, 1, 1])
a1 = a1 / np.linalg.norm(a1)
b1 = np.array([-1, .5, .5])
b1 = b1 / np.linalg.norm(b1)

# target
a2 = np.array([1, 0, 0]) 
a2 = a2 / np.linalg.norm(a2)
b2 = np.array([0, 1, 0]) 
b2 = b2 / np.linalg.norm(b2)

cross_a = np.cross(a1, a2)
cross_a = cross_a / np.linalg.norm(cross_a)
theta_a = angle_between_vectors(a1, a2)

r1 = Rotation.from_rotvec(cross_a * theta_a)


b1_t = b1 @ r1.as_matrix()

cross_b = np.cross(b1_t, b2)
cross_b = cross_b / np.linalg.norm(cross_b)
theta_b = angle_between_vectors(b1_t, b2)

r2 = Rotation.from_rotvec(cross_b * theta_b)

r = r1.as_matrix() @ r2.as_matrix()

a, b = a1 @ r, b1 @ r
# print(bt, b2, angle_between_vectors(bt, b2)/np.pi * 180)
# print(bt @ r_matrix, b2, angle_between_vectors(bt @ r_matrix, b2)/np.pi * 180)
print(a, b)

[ 1.00000000e+00 -2.77555756e-17  0.00000000e+00] [ 0.00000000e+00  1.00000000e+00 -5.55111512e-17]


In [136]:
nx = np.array([1,1,1])
nx = nx / np.linalg.norm(nx)
ny = np.array([-1,.5,.5])
ny = ny / np.linalg.norm(ny)
r = rotate_coordinate_matrix(np.array([nx, ny]), np.array([[1,0,0], [0,1,0]]))

l = np.arange(4*3*2).reshape(4, 3, 2)

# print(
  # np.cross(l[0], l[1]) / 3,
  # np.cross(nx, ny) / np.sqrt(2) * 2
# )
# print(l)
print(l@r)

[[[ 0  1]
  [ 2  3]
  [ 4  5]]

 [[ 6  7]
  [ 8  9]
  [10 11]]

 [[12 13]
  [14 15]
  [16 17]]

 [[18 19]
  [20 21]
  [22 23]]]


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 3 is different from 2)